# 07 - Project Summary & Dashboard

**Residue Manifold Learning (RML) + CGCS for Trisomy 21**  
**Allen Lab Collaboration Overview**

This notebook summarizes the working prototype:

```text
scenario comparison
→ dosage sensitivity
→ intervention recovery
→ next lab-report steps
```


In [ ]:
# ================================================
# SETUP
# ================================================
from pathlib import Path
import sys
import subprocess
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

if (cwd / "src").exists():
    repo_root = cwd
elif cwd.name in {"grok", "chatgpt"} and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]
elif (cwd / REPO_NAME).exists():
    repo_root = cwd / REPO_NAME
else:
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

figures_dir = repo_root / "figures" / "grok"
reports_dir = repo_root / "reports" / "grok"
results_dir = repo_root / "results" / "grok"

for d in [figures_dir, reports_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

USING_SRC = False

try:
    import grok
    from grok.trisomy_metrics import trisomy_cgcs_score, simulate_intervention_recovery

    USING_SRC = True
    version = getattr(grok, "__version__", "unknown")

except Exception as e:
    print("Grok package import failed; using dashboard fallback functions.")
    print("Import error:", repr(e))
    version = "fallback"

    def calculate_cgcs(*components):
        clipped = [max(0.0, min(1.0, float(c))) for c in components]
        product = 1.0
        for c in clipped:
            product *= c
        return product ** (1.0 / len(clipped)) if clipped else 0.0

    def trisomy_cgcs_score(
        dosage_ratio=1.5,
        overexpression_imbalance=0.5,
        global_dysregulation=0.4,
        redundancy_factor=1.0,
        return_components=False,
    ):
        dosage_noise = abs(float(dosage_ratio) - 1.0)
        redundancy_factor = max(float(redundancy_factor), 1e-9)

        components = {
            "dosage_alignment": max(0.0, 1.0 - dosage_noise / redundancy_factor),
            "overexpression_penalty": max(0.0, 1.0 - float(overexpression_imbalance) * dosage_noise),
            "dysregulation_penalty": max(0.0, 1.0 - float(global_dysregulation) * dosage_noise),
            "propagation_alignment": max(0.0, 1.0 - 0.5 * dosage_noise),
        }

        cgcs = calculate_cgcs(*components.values())

        result = {
            "dosage_ratio": float(dosage_ratio),
            "dosage_noise": dosage_noise,
            **components,
            "cgcs": cgcs,
        }

        return result if return_components else {"cgcs": cgcs}

    def simulate_intervention_recovery(baseline_cgcs, recovery_strength=0.45):
        baseline_cgcs = float(baseline_cgcs)
        recovery_strength = max(0.0, min(1.0, float(recovery_strength)))
        return baseline_cgcs + recovery_strength * (1.0 - baseline_cgcs)

print("Project dashboard ready")
print("Repo root:", repo_root)
print("Using src package:", USING_SRC)
print("Version:", version)


## 1. Project Overview

In [ ]:
overview = {
    "framework": "Residue Manifold Learning (RML) + Constraint-Guided Coherence Score (CGCS)",
    "goal": "Quantify chromosomal dosage perturbation and coherence recovery in a Trisomy 21 prototype.",
    "target": "Generate a concise, runnable dashboard for report and collaboration review.",
    "outputs": [
        "scenario comparison table",
        "dashboard figure",
        "intervention recovery table",
        "JSON summary",
        "Markdown summary",
    ],
}

for key, value in overview.items():
    print(f"{key}: {value}")


## 2. Core CGCS Comparison Dashboard

In [ ]:
scenarios = [
    {
        "scenario": "Normal (Euploid)",
        "dosage_ratio": 1.0,
        "overexpression_imbalance": 0.0,
        "global_dysregulation": 0.0,
    },
    {
        "scenario": "Typical Trisomy 21",
        "dosage_ratio": 1.5,
        "overexpression_imbalance": 0.5,
        "global_dysregulation": 0.4,
    },
    {
        "scenario": "High Dysregulation",
        "dosage_ratio": 1.5,
        "overexpression_imbalance": 0.8,
        "global_dysregulation": 0.6,
    },
    {
        "scenario": "Mild Mosaic",
        "dosage_ratio": 1.3,
        "overexpression_imbalance": 0.3,
        "global_dysregulation": 0.2,
    },
]

rows = []
for s in scenarios:
    result = trisomy_cgcs_score(
        dosage_ratio=s["dosage_ratio"],
        overexpression_imbalance=s["overexpression_imbalance"],
        global_dysregulation=s["global_dysregulation"],
        return_components=True,
    )
    rows.append({**s, **result})

df = pd.DataFrame(rows)
display(df.round(4))

scenario_csv = results_dir / "trisomy21_dashboard_scenarios.csv"
df.to_csv(scenario_csv, index=False)
print("Saved:", scenario_csv)


## 3. Dashboard Figure

This figure summarizes the scenario comparison as a compact review surface.


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(df["scenario"], df["cgcs"])
plt.ylim(0, 1.05)
plt.title("CGCS Scenario Comparison for Trisomy 21 Prototype")
plt.ylabel("CGCS score")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)

dashboard_fig = figures_dir / "trisomy21_dashboard_scenarios.png"
plt.savefig(dashboard_fig, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", dashboard_fig)


## 4. Dosage Sensitivity Curve

In [ ]:
dosage_ratios = np.linspace(1.0, 2.0, 41)
dosage_scores = [
    trisomy_cgcs_score(
        dosage_ratio=r,
        overexpression_imbalance=0.5,
        global_dysregulation=0.4,
        return_components=True,
    )["cgcs"]
    for r in dosage_ratios
]

dosage_df = pd.DataFrame({
    "dosage_ratio": dosage_ratios,
    "cgcs": dosage_scores,
})

display(dosage_df.head())
display(dosage_df.tail())

plt.figure(figsize=(10, 5))
plt.plot(dosage_ratios, dosage_scores, "o-", linewidth=2.0)
plt.axvline(1.0, linestyle="--", linewidth=1, label="1.0x normal")
plt.axvline(1.5, linestyle="--", linewidth=1, label="1.5x trisomy")
plt.title("CGCS Dosage Sensitivity")
plt.xlabel("Dosage ratio")
plt.ylabel("CGCS score")
plt.legend()
plt.grid(True, alpha=0.3)

dosage_fig = figures_dir / "trisomy21_dashboard_dosage_sensitivity.png"
plt.savefig(dosage_fig, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", dosage_fig)


## 5. Intervention Recovery Potential

In [ ]:
baseline = trisomy_cgcs_score(
    dosage_ratio=1.5,
    overexpression_imbalance=0.5,
    global_dysregulation=0.4,
    return_components=True,
)["cgcs"]

interventions = [
    {"intervention": "Early Therapies", "recovery_strength": 0.35},
    {"intervention": "Pharmacological Targeting", "recovery_strength": 0.45},
    {"intervention": "Combined Approach", "recovery_strength": 0.60},
]

intervention_rows = []
for item in interventions:
    recovered = simulate_intervention_recovery(
        baseline,
        recovery_strength=item["recovery_strength"],
    )
    intervention_rows.append({
        **item,
        "baseline_cgcs": baseline,
        "recovered_cgcs": recovered,
        "delta_cgcs": recovered - baseline,
    })

intervention_df = pd.DataFrame(intervention_rows)
display(intervention_df.round(4))

intervention_csv = results_dir / "trisomy21_dashboard_interventions.csv"
intervention_df.to_csv(intervention_csv, index=False)
print("Saved:", intervention_csv)


## 6. Key Strengths

- Discrete RML framing for constrained biological variation.
- CGCS gives a compact coherence score for dosage perturbation.
- Scenario comparison supports fast review.
- Recovery simulation gives a first intervention-comparison surface.
- Outputs are saved as figures, CSV, JSON, and Markdown for lab-report reuse.


## 7. Next Steps for Allen Lab Collaboration

1. Replace prototype settings with source-specific values from an Allen Lab dataset or paper.
2. Test person-to-person variability samples.
3. Evaluate specific gene targets or regulatory regions.
4. Add dataset provenance and methods notes.
5. Generate a concise white-paper / lab-report artifact from the dashboard outputs.


## 8. Export Dashboard Summary

In [ ]:
summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "framework": overview["framework"],
    "goal": overview["goal"],
    "version": version,
    "using_src_package": USING_SRC,
    "scenario_records": df.to_dict(orient="records"),
    "intervention_records": intervention_df.to_dict(orient="records"),
    "figures": {
        "scenario_dashboard": str(dashboard_fig.relative_to(repo_root)),
        "dosage_sensitivity": str(dosage_fig.relative_to(repo_root)),
    },
    "results": {
        "scenario_csv": str(scenario_csv.relative_to(repo_root)),
        "intervention_csv": str(intervention_csv.relative_to(repo_root)),
    },
}

summary_json = results_dir / "trisomy21_dashboard_summary.json"
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary_md = reports_dir / "trisomy21_dashboard_summary.md"
summary_md.write_text(
    f"""# Trisomy 21 CGCS Dashboard Summary

**Framework:** {overview['framework']}  
**Goal:** {overview['goal']}  
**Version:** {version}

## Figures

- `{summary['figures']['scenario_dashboard']}`
- `{summary['figures']['dosage_sensitivity']}`

## Results

- `{summary['results']['scenario_csv']}`
- `{summary['results']['intervention_csv']}`

## Key Strengths

- Discrete RML framing for constrained biological variation.
- CGCS gives a compact coherence score for dosage perturbation.
- Scenario comparison supports fast review.
- Recovery simulation gives a first intervention-comparison surface.
- Outputs are saved for lab-report reuse.

## Next Steps

1. Replace prototype settings with source-specific values from an Allen Lab dataset or paper.
2. Test person-to-person variability samples.
3. Evaluate specific gene targets or regulatory regions.
4. Add dataset provenance and methods notes.
5. Generate a concise white-paper / lab-report artifact from the dashboard outputs.
""",
    encoding="utf-8",
)

print("Saved:", summary_json)
print("Saved:", summary_md)
